# CUPED — Variance Reduction in A/B Testing

Demonstrates CUPED (Controlled-experiment Using Pre-Experiment Data) on a
synthetic e-commerce A/B test. CUPED uses pre-experiment behaviour to reduce
the variance of the treatment effect estimate — giving tighter confidence
intervals without collecting more data.

| | |
|---|---|
| **Treatment** | `treatment` — randomly assigned (0 = control, 1 = new feature) |
| **Outcome** | `post_revenue` — revenue per user during the experiment |
| **Covariate** | `pre_revenue` — revenue per user in the 30 days before the experiment |
| **True effect** | +$2.00 per user (small lift we want to detect) |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

RNG = np.random.default_rng(42)

## Generate synthetic data

We simulate 10,000 users. Each user has a latent 'spending propensity' that
drives both their pre-experiment and post-experiment revenue — this is what
creates the correlation that CUPED exploits.

The true treatment effect is **+$2.00**. We want to see whether the standard
A/B test can reliably detect it, and whether CUPED helps.

In [2]:
N           = 10_000   # number of users
TRUE_EFFECT = 2.0      # true ATT in dollars
CORRELATION = 0.7      # pre/post revenue correlation — key CUPED parameter

# Latent spending propensity shared across both periods
propensity = RNG.normal(0, 1, N)

# Pre-experiment revenue: driven by propensity + noise, log-scaled to look like revenue
pre_revenue = np.exp(3 + 0.8 * propensity + RNG.normal(0, 0.5, N))

# Random treatment assignment — 50/50 split
treatment = RNG.binomial(1, 0.5, N)

# Post-experiment revenue: correlated with pre via shared propensity, plus true treatment effect
post_revenue = (
    np.exp(3 + 0.8 * propensity + RNG.normal(0, 0.5, N))
    + TRUE_EFFECT * treatment
)

df = pd.DataFrame({
    "user_id":     np.arange(N),
    "treatment":   treatment,
    "pre_revenue":  pre_revenue,
    "post_revenue": post_revenue,
})

print(f"Users         : {N:,}")
print(f"Treatment     : {treatment.sum():,} ({treatment.mean():.1%})")
print(f"Control       : {(1-treatment).sum():,} ({(1-treatment).mean():.1%})")
print(f"\nPre/post correlation: {df['pre_revenue'].corr(df['post_revenue']):.3f}")
print(f"True treatment effect: ${TRUE_EFFECT:.2f}")

df.describe().T

Users         : 10,000
Treatment     : 5,009 (50.1%)
Control       : 4,991 (49.9%)

Pre/post correlation: 0.622
True treatment effect: $2.00


,count,mean,std,min,25%,50%,75%,max
user_id,10000.0,4999.500000,2886.895680,0.000000,2499.750000,4999.500000,7499.250000,9999.000000
treatment,10000.0,0.500900,0.500024,0.000000,0.000000,1.000000,1.000000,1.000000
pre_revenue,10000.0,31.588047,39.022535,0.615932,10.604089,20.144482,37.348666,865.304336
post_revenue,10000.0,32.355135,38.381582,0.369045,11.374477,20.996269,38.507768,633.146441


In [3]:
df

,user_id,treatment,pre_revenue,post_revenue
0,0,0,27.991816,26.171468
1,1,1,13.702447,6.483654
2,2,0,17.535469,80.336287
3,3,1,36.958760,85.921837
4,4,1,6.383704,4.677226
...,...,...,...,...
9995,9995,1,177.416363,57.475979
9996,9996,0,12.722814,11.858154
9997,9997,0,29.632006,38.925734
9998,9998,0,64.413083,31.942307


In [ ]:
OUT_DIR = Path("../data/raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT_DIR / "ab_test.csv", index=False)
print(f"Saved {len(df):,} rows to {OUT_DIR / 'ab_test.csv'}")